In [4]:
# ============================================================================# MICrONS Visual Decoding with PCA, tPCA + Class Balancing
# Tests combinations of dimensionality reduction and class balancing
# NOW WITH: Classification reports for each configuration

!pip install -q dandi remfile pynwb h5py scikit-learn imbalanced-learn

import numpy as np
import matplotlib.pyplot as plt
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# ============================================================================
# Data Loading Functions
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()

    return nwb


def get_neural_data(nwb):
    """Get the neural recording data."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    rs = fluorescence.roi_response_series["RoiResponseSeries3"]

    print("Total neurons:", rs.data.shape[1])
    print("Total timepoints:", rs.data.shape[0])

    return rs


def get_stimulus_info(nwb):
    """Get information about when stimuli were shown for ALL stimulus types."""
    
    all_starts = []
    all_stops = []
    all_types = []
    
    # Get Clip stimuli (Cinematic, Rendered, sports1m)
    clip_intervals = nwb.intervals['Clip']
    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])
    
    all_starts.extend(clip_starts)
    all_stops.extend(clip_stops)
    all_types.extend(clip_types)
    
    # Get Monet2 stimuli
    monet_intervals = nwb.intervals['Monet2']
    monet_starts = np.array(monet_intervals.start_time[:])
    monet_stops = np.array(monet_intervals.stop_time[:])
    monet_types = np.array(['monet2'] * len(monet_starts))
    
    all_starts.extend(monet_starts)
    all_stops.extend(monet_stops)
    all_types.extend(monet_types)
    
    # Get Trippy stimuli
    trippy_intervals = nwb.intervals['Trippy']
    trippy_starts = np.array(trippy_intervals.start_time[:])
    trippy_stops = np.array(trippy_intervals.stop_time[:])
    trippy_types = np.array(['trippy'] * len(trippy_starts))
    
    all_starts.extend(trippy_starts)
    all_stops.extend(trippy_stops)
    all_types.extend(trippy_types)
    
    # Convert to arrays
    all_starts = np.array(all_starts)
    all_stops = np.array(all_stops)
    all_types = np.array(all_types)
    
    print("\nTotal stimulus presentations:", len(all_starts))
    unique_types = np.unique(all_types)
    for stim_type in unique_types:
        count = np.sum(all_types == stim_type)
        print(f"  {stim_type}: {count} presentations")

    return all_starts, all_stops, all_types


# ============================================================================
# Feature Extraction (Mean and Temporal PCA)
# ============================================================================

def select_best_neurons(rs, timestamps, clip_starts, clip_stops, clip_types,
                       target_conditions, n_select=200):
    """Select the most informative neurons using ANOVA F-test."""
    
    all_neurons = list(range(rs.data.shape[1]))
    X_all, y_all, _ = extract_mean_features(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        all_neurons, target_conditions
    )

    from sklearn.feature_selection import f_classif
    f_scores, p_values = f_classif(X_all, y_all)

    best_indices = np.argsort(f_scores)[-n_select:]
    best_indices = sorted(best_indices.tolist())

    return best_indices


def extract_mean_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                          neuron_list, target_conditions):
    """Extract MEAN firing rate features (standard approach)."""
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    all_trials = []
    all_labels = []

    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]

        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        
        # Just take the mean across time
        neural_features = np.mean(neural_chunk, axis=0)

        all_trials.append(neural_features)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)

    return X, y, condition_names


def extract_temporal_bin_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                                  neuron_list, target_conditions, n_bins=3, max_timepoints=100):
    """
    Extract features from multiple temporal bins.
    Instead of just mean, get mean from early/middle/late parts of response.
    
    This is simpler than tPCA - just divides time into bins and computes
    mean activity in each bin for each neuron.
    
    Args:
        n_bins: Number of time bins to split into (e.g., 3 = early/middle/late)
        max_timepoints: Maximum number of timepoints to use from each trial
    
    Returns:
        X: (n_trials, n_neurons * n_bins) - features for each trial
        y: (n_trials,) - labels
        condition_names: dict mapping condition names to indices
    """
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    all_trials = []
    all_labels = []

    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]

        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        
        # Truncate to max_timepoints
        if neural_chunk.shape[0] > max_timepoints:
            neural_chunk = neural_chunk[:max_timepoints, :]
        
        # Extract temporal bin features
        n_timepoints, n_neurons = neural_chunk.shape
        bin_size = n_timepoints // n_bins
        
        features = []
        for bin_idx in range(n_bins):
            start = bin_idx * bin_size
            end = start + bin_size if bin_idx < n_bins - 1 else n_timepoints
            bin_mean = np.mean(neural_chunk[start:end, :], axis=0)
            features.append(bin_mean)
        
        # Flatten: [early_neuron1, early_neuron2, ..., middle_neuron1, ...]
        neural_features = np.concatenate(features)

        all_trials.append(neural_features)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)
    
    print(f"Temporal bin features shape: {X.shape}")
    print(f"  ({n_bins} bins × {len(neuron_list)} neurons = {X.shape[1]} features)")

    return X, y, condition_names


def extract_tpca_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                         neuron_list, target_conditions, n_components=50, 
                         n_time_windows=4, max_timepoints=100):
    """
    Extract REAL temporal PCA features.
    
    TIME WINDOW APPROACH:
    - Instead of using fine-grained time bins (which creates huge feature spaces),
    - we divide each trial into COARSE time windows (e.g., 4 windows of 25 timepoints each)
    - For each temporal PC, we compute the MEAN activity in each window
    - This gives us: n_components × n_time_windows features per trial
    
    Example with 50 PCs and 4 windows:
    - Window 1 (0-25 timepoints): early response
    - Window 2 (25-50 timepoints): middle response  
    - Window 3 (50-75 timepoints): late response
    - Window 4 (75-100 timepoints): sustained response
    
    Process:
    1. Collect all neural data across all trials: [total_time × n_neurons]
    2. Apply PCA on TIME dimension to find temporal basis functions
    3. Project each trial onto temporal PCs: [time × n_components]
    4. Extract features: mean of each PC in each time window
    """
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    # Step 1: Collect ALL neural data across trials for fitting tPCA
    all_neural_data = []
    
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]
        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]
        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        
        # Truncate to max_timepoints
        if neural_chunk.shape[0] > max_timepoints:
            neural_chunk = neural_chunk[:max_timepoints, :]
        
        all_neural_data.append(neural_chunk)
    
    # Stack all data: [total_timepoints × n_neurons]
    stacked_data = np.vstack(all_neural_data)
    print(f"Stacked data shape: {stacked_data.shape}")
    
    # Step 2: Fit PCA on TIME dimension (across neurons)
    scaler = StandardScaler()
    stacked_data_scaled = scaler.fit_transform(stacked_data)
    
    tpca = PCA(n_components=n_components, random_state=42)
    tpca.fit(stacked_data_scaled)
    
    # Step 3: Project each trial onto temporal PCs and extract features
    window_size = max_timepoints // n_time_windows
    
    all_trials = []
    all_labels = []
    
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]
        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]
        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        
        # Ensure fixed length
        if neural_chunk.shape[0] > max_timepoints:
            neural_chunk = neural_chunk[:max_timepoints, :]
        elif neural_chunk.shape[0] < max_timepoints:
            padding = np.zeros((max_timepoints - neural_chunk.shape[0], neural_chunk.shape[1]))
            neural_chunk = np.vstack([neural_chunk, padding])
        
        # Transform to PC space: [time × neurons] → [time × n_components]
        neural_chunk_scaled = scaler.transform(neural_chunk)
        pc_timecourse = tpca.transform(neural_chunk_scaled)  # [max_timepoints × n_components]
        
        # Step 4: Extract features from PC time courses
        # Strategy: Mean activity in different time windows for each PC
        features = []
        
        for pc_idx in range(n_components):
            for window_idx in range(n_time_windows):
                start = window_idx * window_size
                end = start + window_size if window_idx < n_time_windows - 1 else max_timepoints
                window_mean = np.mean(pc_timecourse[start:end, pc_idx])
                features.append(window_mean)
        
        # Final features: [n_components × n_time_windows]
        all_trials.append(features)
        all_labels.append(condition_names[stim_type])
    
    X = np.array(all_trials)
    y = np.array(all_labels)
    
    print(f"tPCA features shape: {X.shape}")
    
    return X, y, condition_names


# ============================================================================
# Grid search with PCA/tPCA + Class Balancing
# ============================================================================

def grid_search_with_pca_and_balancing(X, y, 
                                       n_components=None,
                                       balancing_strategy='class_weight', 
                                       n_folds=5):
    """
    Use GridSearchCV with PCA (optional) + class balancing strategies.
    """
    
    pca_str = f"PCA({n_components})" if n_components is not None else "No PCA"
    print(f"\nConfiguration: {pca_str} + {balancing_strategy}")
    print(f"Original data shape: {X.shape}")
    print(f"Original class distribution: {Counter(y)}")
    
    # Build pipeline based on strategy
    if balancing_strategy == 'class_weight':
        if n_components is None:
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
            ])
        else:
            pipeline = Pipeline([
                ('pca', PCA(n_components=n_components, random_state=42)),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
            ])
        
        param_grid = [
            {
                'classifier__C': [0.001, 0.01, 0.1, 1.0, 10],
                'classifier__penalty': ['l2'],
                'classifier__solver': ['lbfgs'],
                'classifier__max_iter': [5000]
            },
            {
                'classifier__C': [0.001, 0.01, 0.1, 1.0, 10],
                'classifier__penalty': ['l1'],
                'classifier__solver': ['saga'],
                'classifier__max_iter': [5000]
            }
        ]
        
    elif balancing_strategy in ['undersample', 'oversample']:
        sampler = RandomUnderSampler(random_state=42) if balancing_strategy == 'undersample' else RandomOverSampler(random_state=42)
        
        if n_components is None:
            pipeline = ImbPipeline([
                ('sampler', sampler),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42))
            ])
        else:
            pipeline = ImbPipeline([
                ('pca', PCA(n_components=n_components, random_state=42)),
                ('sampler', sampler),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42))
            ])
        
        param_grid = [
            {
                'classifier__C': [0.001, 0.01, 0.1, 1.0, 10],
                'classifier__penalty': ['l2'],
                'classifier__solver': ['lbfgs'],
                'classifier__max_iter': [5000]
            }
        ]
    
    else:
        raise ValueError(f"Unknown strategy: {balancing_strategy}")
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='balanced_accuracy',
        n_jobs=-1,
        verbose=0,
        refit=True
    )

    grid_search.fit(X, y)
    
    # Report PCA variance explained if used
    if n_components is not None and hasattr(grid_search.best_estimator_.named_steps.get('pca', None), 'explained_variance_ratio_'):
        pca_step = grid_search.best_estimator_.named_steps['pca']
        var_explained = np.sum(pca_step.explained_variance_ratio_)
        print(f"PCA: {pca_step.n_components_} components explain {var_explained*100:.2f}% variance")

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_


def evaluate_best_model(best_pipeline, X, y, condition_names, n_folds=5):
    """Evaluate the best model with detailed metrics and classification report."""
    from sklearn.base import clone
    from sklearn.metrics import balanced_accuracy_score, classification_report
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = []
    fold_balanced_accuracies = []
    all_y_true = []
    all_y_pred = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipeline = clone(best_pipeline)
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        
        accuracy = np.mean(y_test == y_pred)
        balanced_acc = balanced_accuracy_score(y_test, y_pred)
        
        fold_accuracies.append(accuracy)
        fold_balanced_accuracies.append(balanced_acc)
        
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)

    # Generate classification report
    class_names = [k for k, v in sorted(condition_names.items(), key=lambda x: x[1])]
    
    print("\nClassification Report (aggregated across folds):")
    print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=2))
    
    print(f"\nAccuracy: {np.mean(fold_accuracies)*100:.2f}% ± {np.std(fold_accuracies)*100:.2f}%")
    print(f"Balanced Accuracy: {np.mean(fold_balanced_accuracies)*100:.2f}% ± {np.std(fold_balanced_accuracies)*100:.2f}%")
    
    return fold_accuracies, fold_balanced_accuracies


# ============================================================================
# Main Analysis
# ============================================================================

print("="*70)
print("Neural Decoding: PCA vs tPCA + Class Balancing")
print("="*70)

# Load data
nwb = connect_to_data()
rs = get_neural_data(nwb)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)

timestamps = np.array(rs.timestamps[:100000])

# All 5 stimulus types
target_conditions_all = ['Cinematic', 'Rendered', 'sports1m', 'monet2', 'trippy']

print("\n" + "="*70)
print("Selecting best 200 neurons...")
print("="*70)

neuron_list_all = select_best_neurons(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    target_conditions_all, n_select=200
)

# ============================================================================
# EXPERIMENT 1: Mean Features + PCA
# ============================================================================

print("\n" + "="*70)
print("EXPERIMENT 1: Mean Features + Regular PCA")
print("="*70)

X_mean, y_mean, condition_names_mean = extract_mean_features(
    rs, timestamps, clip_starts, clip_stops, clip_types,
    neuron_list_all, target_conditions_all
)

print(f"\nMean features shape: {X_mean.shape}")

# Test only the best configurations to save time
strategies = ['class_weight']
pca_options = [None, 50, 100]
results_mean = {}

for strategy in strategies:
    for n_pca in pca_options:
        config_name = f"{strategy}_PCA{n_pca}" if n_pca else f"{strategy}_NoPCA"
        
        print(f"\n{'='*70}")
        print(f"Testing: {config_name}")
        print(f"{'='*70}")
        
        best_pipeline, best_params, best_score = grid_search_with_pca_and_balancing(
            X_mean, y_mean, 
            n_components=n_pca,
            balancing_strategy=strategy, 
            n_folds=5
        )
        
        fold_accs, fold_bal_accs = evaluate_best_model(
            best_pipeline, X_mean, y_mean, condition_names_mean, n_folds=5
        )
        
        results_mean[config_name] = {
            'strategy': strategy,
            'n_pca': n_pca,
            'accuracy': np.mean(fold_accs),
            'accuracy_std': np.std(fold_accs),
            'balanced_accuracy': np.mean(fold_bal_accs),
            'balanced_accuracy_std': np.std(fold_bal_accs),
            'best_params': best_params
        }


# ============================================================================
# EXPERIMENT 2: Temporal Features + tPCA
# ============================================================================

print("\n" + "="*70)
print("EXPERIMENT 2: Temporal Features + REAL tPCA")
print("="*70)

# Test different configurations
tpca_component_options = [30, 50]
time_window_options = [2, 4, 8]

results_tpca = {}

for n_tpcs in tpca_component_options:
    for n_windows in time_window_options:
        config_name = f"tPCA{n_tpcs}_win{n_windows}"
        
        print(f"\n{'='*70}")
        print(f"Testing: {config_name}")
        print(f"{'='*70}")
        
        # Extract features with this configuration
        X_tpca_temp, y_tpca_temp, condition_names_tpca = extract_tpca_features(
            rs, timestamps, clip_starts, clip_stops, clip_types,
            neuron_list_all, target_conditions_all, 
            n_components=n_tpcs,
            n_time_windows=n_windows,
            max_timepoints=100
        )
        
        strategy = 'class_weight'
        
        best_pipeline, best_params, best_score = grid_search_with_pca_and_balancing(
            X_tpca_temp, y_tpca_temp, 
            n_components=None,
            balancing_strategy=strategy, 
            n_folds=5
        )
        
        fold_accs, fold_bal_accs = evaluate_best_model(
            best_pipeline, X_tpca_temp, y_tpca_temp, condition_names_tpca, n_folds=5
        )
        
        results_tpca[config_name] = {
            'n_tpcs': n_tpcs,
            'n_windows': n_windows,
            'accuracy': np.mean(fold_accs),
            'accuracy_std': np.std(fold_accs),
            'balanced_accuracy': np.mean(fold_bal_accs),
            'balanced_accuracy_std': np.std(fold_bal_accs),
            'best_params': best_params
        }


# ============================================================================
# Results Summary
# ============================================================================

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print("\nEXPERIMENT 1: Mean Features + PCA")
print("-"*70)
for config_name, result in sorted(results_mean.items()):
    print(f"\n{config_name}:")
    print(f"  Balanced Accuracy: {result['balanced_accuracy']*100:.2f}% ± {result['balanced_accuracy_std']*100:.2f}%")

print("\n\nEXPERIMENT 2: Temporal Features + tPCA")
print("-"*70)
for config_name, result in sorted(results_tpca.items()):
    print(f"\n{config_name}:")
    print(f"  Balanced Accuracy: {result['balanced_accuracy']*100:.2f}% ± {result['balanced_accuracy_std']*100:.2f}%")

best_mean = max(results_mean.items(), key=lambda x: x[1]['balanced_accuracy'])
best_tpca = max(results_tpca.items(), key=lambda x: x[1]['balanced_accuracy'])

print("\n" + "="*70)
print("BEST CONFIGURATIONS")
print("="*70)
print(f"\nBest Mean Features: {best_mean[0]}")
print(f"  Balanced Accuracy: {best_mean[1]['balanced_accuracy']*100:.2f}%")

print(f"\nBest Temporal Features: {best_tpca[0]}")
print(f"  Balanced Accuracy: {best_tpca[1]['balanced_accuracy']*100:.2f}%")

improvement = (best_tpca[1]['balanced_accuracy'] - best_mean[1]['balanced_accuracy']) * 100
print(f"\nImprovement: {improvement:+.2f}%")

Neural Decoding: PCA vs tPCA + Class Balancing
Total neurons: 1455
Total timepoints: 40000

Total stimulus presentations: 464
  Cinematic: 128 presentations
  Rendered: 128 presentations
  monet2: 40 presentations
  sports1m: 128 presentations
  trippy: 40 presentations

Selecting best 200 neurons...

EXPERIMENT 1: Mean Features + Regular PCA

Mean features shape: (464, 200)

Testing: class_weight_NoPCA

Configuration: No PCA + class_weight
Original data shape: (464, 200)
Original class distribution: Counter({np.int64(2): 128, np.int64(0): 128, np.int64(1): 128, np.int64(3): 40, np.int64(4): 40})

Classification Report (aggregated across folds):
              precision    recall  f1-score   support

   Cinematic       0.65      0.66      0.65       128
    Rendered       0.66      0.60      0.63       128
    sports1m       0.65      0.67      0.66       128
      monet2       0.95      1.00      0.98        40
      trippy       0.86      0.95      0.90        40

    accuracy        

In [7]:
# ============================================================================
# MICrONS Visual Decoding with PCA, tPCA + Class Balancing
# Tests combinations of dimensionality reduction and class balancing
!pip install -q dandi remfile pynwb h5py scikit-learn imbalanced-learn

import numpy as np
import matplotlib.pyplot as plt
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# ============================================================================
# Data Loading Functions
# ============================================================================

def connect_to_data():
    """Connect to the dataset and return the nwb file."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset("000402", "draft")
    all_assets = dandiset.get_assets()

    asset = None
    for a in all_assets:
        if a.path.endswith(".nwb"):
            asset = a
            break

    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()

    return nwb


def get_neural_data(nwb):
    """Get the neural recording data."""
    ophys = nwb.processing["ophys"]
    fluorescence = ophys.data_interfaces["Fluorescence"]
    
    # Get all ROI response series
    all_roi_series = {}
    for roi_name in fluorescence.roi_response_series.keys():
        rs = fluorescence.roi_response_series[roi_name]
        all_roi_series[roi_name] = rs
        print(f"  {roi_name}: {rs.data.shape[1]} neurons, {rs.data.shape[0]} timepoints")
    
    print(f"\nFound {len(all_roi_series)} ROI response series")
    print(f"Total neurons across all series: {sum(rs.data.shape[1] for rs in all_roi_series.values())}")

    return all_roi_series


def get_stimulus_info(nwb):
    """Get information about when stimuli were shown for ALL stimulus types."""
    
    all_starts = []
    all_stops = []
    all_types = []
    
    # Get Clip stimuli (Cinematic, Rendered, sports1m)
    clip_intervals = nwb.intervals['Clip']
    clip_starts = np.array(clip_intervals.start_time[:])
    clip_stops = np.array(clip_intervals.stop_time[:])
    clip_types = np.array(clip_intervals.short_movie_name[:])
    
    all_starts.extend(clip_starts)
    all_stops.extend(clip_stops)
    all_types.extend(clip_types)
    
    # Get Monet2 stimuli
    monet_intervals = nwb.intervals['Monet2']
    monet_starts = np.array(monet_intervals.start_time[:])
    monet_stops = np.array(monet_intervals.stop_time[:])
    monet_types = np.array(['monet2'] * len(monet_starts))
    
    all_starts.extend(monet_starts)
    all_stops.extend(monet_stops)
    all_types.extend(monet_types)
    
    # Get Trippy stimuli
    trippy_intervals = nwb.intervals['Trippy']
    trippy_starts = np.array(trippy_intervals.start_time[:])
    trippy_stops = np.array(trippy_intervals.stop_time[:])
    trippy_types = np.array(['trippy'] * len(trippy_starts))
    
    all_starts.extend(trippy_starts)
    all_stops.extend(trippy_stops)
    all_types.extend(trippy_types)
    
    # Convert to arrays
    all_starts = np.array(all_starts)
    all_stops = np.array(all_stops)
    all_types = np.array(all_types)
    
    print("\nTotal stimulus presentations:", len(all_starts))
    unique_types = np.unique(all_types)
    for stim_type in unique_types:
        count = np.sum(all_types == stim_type)
        print(f"  {stim_type}: {count} presentations")

    return all_starts, all_stops, all_types


# ============================================================================
# Feature Extraction (Mean and Temporal PCA)
# ============================================================================

def select_best_neurons_from_roi(rs, timestamps, clip_starts, clip_stops, clip_types,
                                 target_conditions, n_select=150):
    """Select the most informative neurons from a single ROI using ANOVA F-test."""
    
    all_neurons = list(range(rs.data.shape[1]))
    X_all, y_all, _ = extract_mean_features(
        rs, timestamps, clip_starts, clip_stops, clip_types,
        all_neurons, target_conditions
    )

    from sklearn.feature_selection import f_classif
    f_scores, p_values = f_classif(X_all, y_all)

    best_indices = np.argsort(f_scores)[-n_select:]
    best_indices = sorted(best_indices.tolist())
    
    # Return indices with their F-scores for tracking
    best_f_scores = f_scores[best_indices]

    return best_indices, best_f_scores


def select_best_neurons_across_all_rois(all_roi_series, timestamps, clip_starts, clip_stops, 
                                        clip_types, target_conditions, 
                                        n_per_roi=150, n_final=200):
    """
    Multi-stage neuron selection:
    1. For each ROI, select top n_per_roi neurons
    2. Combine all selected neurons
    3. Do final selection to get top n_final neurons overall
    
    Returns:
        selected_neurons: list of (roi_name, neuron_idx) tuples
    """
    
    print("\n" + "="*70)
    print("STAGE 1: Selecting best neurons from each ROI")
    print("="*70)
    
    all_selected_neurons = []
    
    for roi_name, rs in all_roi_series.items():
        print(f"\nProcessing {roi_name} ({rs.data.shape[1]} neurons)...")
        
        best_indices, f_scores = select_best_neurons_from_roi(
            rs, timestamps, clip_starts, clip_stops, clip_types,
            target_conditions, n_select=min(n_per_roi, rs.data.shape[1])
        )
        
        print(f"  Selected {len(best_indices)} neurons")
        print(f"  F-score range: {f_scores.min():.2f} - {f_scores.max():.2f}")
        
        # Store as (roi_name, neuron_idx, f_score) tuples
        for idx, score in zip(best_indices, f_scores):
            all_selected_neurons.append((roi_name, idx, score))
    
    print(f"\n" + "="*70)
    print(f"STAGE 2: Final selection from {len(all_selected_neurons)} neurons")
    print("="*70)
    
    # Extract features from all selected neurons combined
    print("\nExtracting features from all selected neurons...")
    
    # Build combined feature matrix
    all_trial_features = []
    all_labels = []
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]
        if stim_type not in target_conditions:
            continue
            
        start_time = clip_starts[i]
        stop_time = clip_stops[i]
        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)
        
        # Collect features from each ROI
        trial_features = []
        for roi_name, neuron_idx, _ in all_selected_neurons:
            rs = all_roi_series[roi_name]
            neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_idx])
            trial_features.append(np.mean(neural_chunk))
        
        all_trial_features.append(trial_features)
        all_labels.append(condition_names[stim_type])
    
    X_combined = np.array(all_trial_features)
    y_combined = np.array(all_labels)
    
    print(f"Combined feature matrix shape: {X_combined.shape}")
    
    # Do final feature selection
    from sklearn.feature_selection import f_classif
    f_scores_combined, _ = f_classif(X_combined, y_combined)
    
    best_final_indices = np.argsort(f_scores_combined)[-n_final:]
    best_final_indices = sorted(best_final_indices.tolist())
    
    final_selected_neurons = [all_selected_neurons[i] for i in best_final_indices]
    
    print(f"\nFinal selection: {len(final_selected_neurons)} neurons")
    
    # Count neurons per ROI
    roi_counts = {}
    for roi_name, _, _ in final_selected_neurons:
        roi_counts[roi_name] = roi_counts.get(roi_name, 0) + 1
    
    print("\nNeurons per ROI:")
    for roi_name, count in sorted(roi_counts.items()):
        print(f"  {roi_name}: {count} neurons")
    
    return final_selected_neurons


def extract_mean_features(rs, timestamps, clip_starts, clip_stops, clip_types,
                          neuron_list, target_conditions):
    """Extract MEAN firing rate features (standard approach)."""
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    all_trials = []
    all_labels = []

    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]

        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_list])
        
        # Just take the mean across time
        neural_features = np.mean(neural_chunk, axis=0)

        all_trials.append(neural_features)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)

    return X, y, condition_names


def extract_mean_features_multi_roi(all_roi_series, selected_neurons, timestamps, 
                                    clip_starts, clip_stops, clip_types, target_conditions):
    """Extract MEAN features from neurons across multiple ROIs."""
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    all_trials = []
    all_labels = []

    for i in range(len(clip_starts)):
        stim_type = clip_types[i]

        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]

        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        # Collect features from each selected neuron
        trial_features = []
        for roi_name, neuron_idx, _ in selected_neurons:
            rs = all_roi_series[roi_name]
            neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_idx])
            trial_features.append(np.mean(neural_chunk))
        
        all_trials.append(trial_features)
        all_labels.append(condition_names[stim_type])

    X = np.array(all_trials)
    y = np.array(all_labels)

    return X, y, condition_names


def extract_tpca_features_multi_roi(all_roi_series, selected_neurons, timestamps, 
                                    clip_starts, clip_stops, clip_types,
                                    target_conditions, n_components=50, 
                                    n_time_windows=4, max_timepoints=100):
    """
    Extract REAL temporal PCA features from multi-ROI selected neurons.
    
    TIME WINDOW APPROACH:
    - Instead of using fine-grained time bins (which creates huge feature spaces),
    - we divide each trial into COARSE time windows (e.g., 4 windows of 25 timepoints each)
    - For each temporal PC, we compute the MEAN activity in each window
    - This gives us: n_components × n_time_windows features per trial
    
    Process:
    1. Collect all neural data across all trials: [total_time × n_neurons]
    2. Apply PCA on TIME dimension to find temporal basis functions
    3. Project each trial onto temporal PCs: [time × n_components]
    4. Extract features: mean of each PC in each time window
    """
    
    condition_names = {cond: i for i, cond in enumerate(target_conditions)}
    
    # Step 1: Collect ALL neural data across trials for fitting tPCA
    all_neural_data = []
    
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]
        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]
        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        # Collect data from each selected neuron
        trial_data = []
        for roi_name, neuron_idx, _ in selected_neurons:
            rs = all_roi_series[roi_name]
            neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_idx])
            
            # Ensure fixed length
            if neural_chunk.shape[0] > max_timepoints:
                neural_chunk = neural_chunk[:max_timepoints]
            elif neural_chunk.shape[0] < max_timepoints:
                padding = np.zeros(max_timepoints - neural_chunk.shape[0])
                neural_chunk = np.concatenate([neural_chunk, padding])
            
            trial_data.append(neural_chunk)
        
        # Stack across neurons: [max_timepoints × n_selected_neurons]
        trial_data = np.column_stack(trial_data)
        all_neural_data.append(trial_data)
    
    # Stack all data: [total_timepoints × n_neurons]
    stacked_data = np.vstack(all_neural_data)
    print(f"Stacked data shape: {stacked_data.shape}")
    
    # Step 2: Fit PCA on TIME dimension (across neurons)
    scaler = StandardScaler()
    stacked_data_scaled = scaler.fit_transform(stacked_data)
    
    tpca = PCA(n_components=n_components, random_state=42)
    tpca.fit(stacked_data_scaled)
    
    print(f"tPCA: {n_components} components explain {np.sum(tpca.explained_variance_ratio_)*100:.2f}% variance")
    
    # Step 3: Project each trial onto temporal PCs and extract features
    window_size = max_timepoints // n_time_windows
    
    all_trials = []
    all_labels = []
    
    for i in range(len(clip_starts)):
        stim_type = clip_types[i]
        if stim_type not in target_conditions:
            continue

        start_time = clip_starts[i]
        stop_time = clip_stops[i]
        start_idx = np.searchsorted(timestamps, start_time)
        stop_idx = np.searchsorted(timestamps, stop_time)

        # Collect data from each selected neuron
        trial_data = []
        for roi_name, neuron_idx, _ in selected_neurons:
            rs = all_roi_series[roi_name]
            neural_chunk = np.array(rs.data[start_idx:stop_idx, neuron_idx])
            
            # Ensure fixed length
            if neural_chunk.shape[0] > max_timepoints:
                neural_chunk = neural_chunk[:max_timepoints]
            elif neural_chunk.shape[0] < max_timepoints:
                padding = np.zeros(max_timepoints - neural_chunk.shape[0])
                neural_chunk = np.concatenate([neural_chunk, padding])
            
            trial_data.append(neural_chunk)
        
        # Stack across neurons: [max_timepoints × n_selected_neurons]
        trial_matrix = np.column_stack(trial_data)
        
        # Transform to PC space: [time × neurons] → [time × n_components]
        trial_matrix_scaled = scaler.transform(trial_matrix)
        pc_timecourse = tpca.transform(trial_matrix_scaled)
        
        # Step 4: Extract features from PC time courses
        features = []
        for pc_idx in range(n_components):
            for window_idx in range(n_time_windows):
                start = window_idx * window_size
                end = start + window_size if window_idx < n_time_windows - 1 else max_timepoints
                window_mean = np.mean(pc_timecourse[start:end, pc_idx])
                features.append(window_mean)
        
        all_trials.append(features)
        all_labels.append(condition_names[stim_type])
    
    X = np.array(all_trials)
    y = np.array(all_labels)
    
    print(f"tPCA features shape: {X.shape}")
    print(f"  ({n_components} PCs × {n_time_windows} windows = {X.shape[1]} features)")
    
    return X, y, condition_names


# ============================================================================
# Grid search with PCA/tPCA + Class Balancing
# ============================================================================

def grid_search_with_pca_and_balancing(X, y, 
                                       n_components=None,
                                       balancing_strategy='class_weight', 
                                       n_folds=5):
    """
    Use GridSearchCV with PCA (optional) + class balancing strategies.
    """
    
    pca_str = f"Additional PCA({n_components})" if n_components is not None else "No additional PCA"
    print(f"\nConfiguration: {pca_str} + {balancing_strategy}")
    print(f"Input data shape: {X.shape}")
    print(f"Class distribution: {Counter(y)}")
    
    # Build pipeline based on strategy
    if balancing_strategy == 'class_weight':
        if n_components is None:
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
            ])
        else:
            pipeline = Pipeline([
                ('pca', PCA(n_components=n_components, random_state=42)),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
            ])
        
        param_grid = [
            {
                'classifier__C': [0.001, 0.01, 0.1, 1.0, 10],
                'classifier__penalty': ['l2'],
                'classifier__solver': ['lbfgs'],
                'classifier__max_iter': [5000]
            },
            {
                'classifier__C': [0.001, 0.01, 0.1, 1.0, 10],
                'classifier__penalty': ['l1'],
                'classifier__solver': ['saga'],
                'classifier__max_iter': [5000]
            }
        ]
        
    elif balancing_strategy in ['undersample', 'oversample']:
        sampler = RandomUnderSampler(random_state=42) if balancing_strategy == 'undersample' else RandomOverSampler(random_state=42)
        
        if n_components is None:
            pipeline = ImbPipeline([
                ('sampler', sampler),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42))
            ])
        else:
            pipeline = ImbPipeline([
                ('pca', PCA(n_components=n_components, random_state=42)),
                ('sampler', sampler),
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42))
            ])
        
        param_grid = [
            {
                'classifier__C': [0.001, 0.01, 0.1, 1.0, 10],
                'classifier__penalty': ['l2'],
                'classifier__solver': ['lbfgs'],
                'classifier__max_iter': [5000]
            }
        ]
    
    else:
        raise ValueError(f"Unknown strategy: {balancing_strategy}")
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='balanced_accuracy',
        n_jobs=-1,
        verbose=0,
        refit=True
    )

    grid_search.fit(X, y)
    
    # Report PCA variance explained if used
    if n_components is not None and hasattr(grid_search.best_estimator_.named_steps.get('pca', None), 'explained_variance_ratio_'):
        pca_step = grid_search.best_estimator_.named_steps['pca']
        var_explained = np.sum(pca_step.explained_variance_ratio_)
        print(f"PCA: {pca_step.n_components_} components explain {var_explained*100:.2f}% variance")

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_


def evaluate_best_model(best_pipeline, X, y, condition_names, n_folds=5):
    """Evaluate the best model with detailed metrics and classification report."""
    from sklearn.base import clone
    from sklearn.metrics import balanced_accuracy_score, classification_report
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = []
    fold_balanced_accuracies = []
    all_y_true = []
    all_y_pred = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipeline = clone(best_pipeline)
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        
        accuracy = np.mean(y_test == y_pred)
        balanced_acc = balanced_accuracy_score(y_test, y_pred)
        
        fold_accuracies.append(accuracy)
        fold_balanced_accuracies.append(balanced_acc)
        
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)

    # Generate classification report
    class_names = [k for k, v in sorted(condition_names.items(), key=lambda x: x[1])]
    
    print("\nClassification Report (aggregated across folds):")
    print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=2))
    
    print(f"\nAccuracy: {np.mean(fold_accuracies)*100:.2f}% ± {np.std(fold_accuracies)*100:.2f}%")
    print(f"Balanced Accuracy: {np.mean(fold_balanced_accuracies)*100:.2f}% ± {np.std(fold_balanced_accuracies)*100:.2f}%")
    
    return fold_accuracies, fold_balanced_accuracies


# ============================================================================
# Main Analysis
# ============================================================================

print("="*70)
print("Neural Decoding: PCA vs tPCA + Class Balancing")
print("="*70)

# Load data
nwb = connect_to_data()
all_roi_series = get_neural_data(nwb)
clip_starts, clip_stops, clip_types = get_stimulus_info(nwb)

timestamps = np.array(list(all_roi_series.values())[0].timestamps[:100000])

# All 5 stimulus types
target_conditions_all = ['Cinematic', 'Rendered', 'sports1m', 'monet2', 'trippy']

print("\n" + "="*70)
print("Multi-ROI Neuron Selection")
print("="*70)

selected_neurons = select_best_neurons_across_all_rois(
    all_roi_series, timestamps, clip_starts, clip_stops, clip_types,
    target_conditions_all, n_per_roi=150, n_final=200
)

# ============================================================================
# EXPERIMENT 1: Mean Features + PCA
# ============================================================================

print("\n" + "="*70)
print("EXPERIMENT 1: Mean Features + Regular PCA")
print("="*70)

X_mean, y_mean, condition_names_mean = extract_mean_features_multi_roi(
    all_roi_series, selected_neurons, timestamps, clip_starts, clip_stops, clip_types,
    target_conditions_all
)

print(f"\nMean features shape: {X_mean.shape}")

# Test only the best configurations to save time
strategies = ['class_weight']
pca_options = [None, 50, 100]
results_mean = {}

for strategy in strategies:
    for n_pca in pca_options:
        config_name = f"{strategy}_PCA{n_pca}" if n_pca else f"{strategy}_NoPCA"
        
        print(f"\n{'='*70}")
        print(f"Testing: {config_name}")
        print(f"{'='*70}")
        
        best_pipeline, best_params, best_score = grid_search_with_pca_and_balancing(
            X_mean, y_mean, 
            n_components=n_pca,
            balancing_strategy=strategy, 
            n_folds=5
        )
        
        fold_accs, fold_bal_accs = evaluate_best_model(
            best_pipeline, X_mean, y_mean, condition_names_mean, n_folds=5
        )
        
        results_mean[config_name] = {
            'strategy': strategy,
            'n_pca': n_pca,
            'accuracy': np.mean(fold_accs),
            'accuracy_std': np.std(fold_accs),
            'balanced_accuracy': np.mean(fold_bal_accs),
            'balanced_accuracy_std': np.std(fold_bal_accs),
            'best_params': best_params
        }


# ============================================================================
# EXPERIMENT 2: Temporal Features + tPCA
# ============================================================================

print("\n" + "="*70)
print("EXPERIMENT 2: Temporal Features + REAL tPCA")
print("="*70)

# Test different configurations
tpca_component_options = [30, 50]
time_window_options = [2, 4, 8]

results_tpca = {}

for n_tpcs in tpca_component_options:
    for n_windows in time_window_options:
        config_name = f"tPCA{n_tpcs}_win{n_windows}"
        
        print(f"\n{'='*70}")
        print(f"Testing: {config_name}")
        print(f"{'='*70}")
        
        # FIXED: Use the multi-ROI version
        X_tpca_temp, y_tpca_temp, condition_names_tpca = extract_tpca_features_multi_roi(
            all_roi_series, selected_neurons, timestamps, 
            clip_starts, clip_stops, clip_types,
            target_conditions_all, 
            n_components=n_tpcs,
            n_time_windows=n_windows,
            max_timepoints=100
        )
        
        strategy = 'class_weight'
        
        best_pipeline, best_params, best_score = grid_search_with_pca_and_balancing(
            X_tpca_temp, y_tpca_temp, 
            n_components=None,
            balancing_strategy=strategy, 
            n_folds=5
        )
        
        fold_accs, fold_bal_accs = evaluate_best_model(
            best_pipeline, X_tpca_temp, y_tpca_temp, condition_names_tpca, n_folds=5
        )
        
        results_tpca[config_name] = {
            'n_tpcs': n_tpcs,
            'n_windows': n_windows,
            'accuracy': np.mean(fold_accs),
            'accuracy_std': np.std(fold_accs),
            'balanced_accuracy': np.mean(fold_bal_accs),
            'balanced_accuracy_std': np.std(fold_bal_accs),
            'best_params': best_params
        }
# ============================================================================
# Results Summary
# ============================================================================

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print("\nEXPERIMENT 1: Mean Features + PCA")
print("-"*70)
for config_name, result in sorted(results_mean.items()):
    print(f"\n{config_name}:")
    print(f"  Balanced Accuracy: {result['balanced_accuracy']*100:.2f}% ± {result['balanced_accuracy_std']*100:.2f}%")

print("\n\nEXPERIMENT 2: Temporal Features + tPCA")
print("-"*70)
for config_name, result in sorted(results_tpca.items()):
    print(f"\n{config_name}:")
    print(f"  Balanced Accuracy: {result['balanced_accuracy']*100:.2f}% ± {result['balanced_accuracy_std']*100:.2f}%")

best_mean = max(results_mean.items(), key=lambda x: x[1]['balanced_accuracy'])
best_tpca = max(results_tpca.items(), key=lambda x: x[1]['balanced_accuracy'])

print("\n" + "="*70)
print("BEST CONFIGURATIONS")
print("="*70)
print(f"\nBest Mean Features: {best_mean[0]}")
print(f"  Balanced Accuracy: {best_mean[1]['balanced_accuracy']*100:.2f}%")

print(f"\nBest Temporal Features: {best_tpca[0]}")
print(f"  Balanced Accuracy: {best_tpca[1]['balanced_accuracy']*100:.2f}%")

improvement = (best_tpca[1]['balanced_accuracy'] - best_mean[1]['balanced_accuracy']) * 100
print(f"\nImprovement: {improvement:+.2f}%")


Neural Decoding: PCA vs tPCA + Class Balancing
  RoiResponseSeries1: 643 neurons, 40000 timepoints
  RoiResponseSeries2: 452 neurons, 40000 timepoints
  RoiResponseSeries3: 1455 neurons, 40000 timepoints
  RoiResponseSeries4: 1389 neurons, 40000 timepoints
  RoiResponseSeries5: 1420 neurons, 40000 timepoints
  RoiResponseSeries6: 1411 neurons, 40000 timepoints
  RoiResponseSeries7: 895 neurons, 40000 timepoints
  RoiResponseSeries8: 730 neurons, 40000 timepoints

Found 8 ROI response series
Total neurons across all series: 8395

Total stimulus presentations: 464
  Cinematic: 128 presentations
  Rendered: 128 presentations
  monet2: 40 presentations
  sports1m: 128 presentations
  trippy: 40 presentations

Multi-ROI Neuron Selection

STAGE 1: Selecting best neurons from each ROI

Processing RoiResponseSeries1 (643 neurons)...
  Selected 150 neurons
  F-score range: 5.78 - 146.42

Processing RoiResponseSeries2 (452 neurons)...
  Selected 150 neurons
  F-score range: 5.47 - 125.09

Proces